## Local Agent with LMStudio and Cogito v1

In this example we will learn how to use fully local LLMs, implementing the new cogito v1 models (specifically `cogito-v1-preview-qwen-32b`). These models are incredibly competent LLMs fully competent of powering agentic workflows with tool calling, while being small enough to run on personal hardware.

We will be using LM Studio to host Cogito models locally, installation instructions can be found [here](https://lmstudio.ai/download).

LM Studio (on Mac) supports all GGUF quantized models. That means that we must use [`lmstudio-community/cogito-v1-preview-qwen-32b`](https://huggingface.co/lmstudio-community/cogito-v1-preview-qwen-32b) which can be downloaded for LM Studio [here](https://model.lmstudio.ai/download/lmstudio-community/cogito-v1-preview-qwen-32b).

## Using Cogito v1

Once the model has been downloaded we can select **Start server on port 1234** in our LM Studio interface (accessible by clicking the icon in the taskbar) and load our chosen model. Then we confirm LM Studio is accessible like so:

In [1]:
!curl http://localhost:1234/v1/models

{
  "data": [
    {
      "id": "cogito-v1-preview-qwen-32b",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "cogito-v1-preview-llama-70b",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "unsloth/llama-4-scout-17b-16e-instruct",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "lmstudio-community/llama-4-scout-17b-16e-instruct",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "text-embedding-nomic-embed-text-v1.5",
      "object": "model",
      "owned_by": "organization_owner"
    },
    {
      "id": "mistral-small-3.1-24b-instruct-2503",
      "object": "model",
      "owned_by": "organization_owner"
    }
  ],
  "object": "list"
}

We can take the model name from above and insert it into our `model` parameter below (after `lm_studio/`):

In [2]:
from litellm import completion
import os

MODEL = "lm_studio/cogito-v1-preview-qwen-32b"

# set to the port LM studio is using, default is 1234
os.environ["LM_STUDIO_API_BASE"] = "http://localhost:1234/v1"

response = completion(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key"  # need a dummy API key
)
print(response)

ModelResponse(id='chatcmpl-71yxo06zqcihet9z3h6ie', created=1744987223, model='lm_studio/cogito-v1-preview-qwen-32b', object='chat.completion', system_fingerprint='cogito-v1-preview-qwen-32b', choices=[Choices(finish_reason='stop', index=0, message=Message(content="I'm doing well, thank you for asking! How can I help you today?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None}))], usage=Usage(completion_tokens=17, prompt_tokens=14, total_tokens=31, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, stats={})


## Async Streaming

Let's see how we can implement asynchronous streaming via LiteLLM.

In [3]:
from litellm import acompletion

response = await acompletion(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    stream=True
)

async for chunk in response:
    print(chunk)

ModelResponseStream(id='chatcmpl-1f1736cf-d96e-43e8-88a1-96562f8746e9', created=1744987308, model='cogito-v1-preview-qwen-32b', object='chat.completion.chunk', system_fingerprint='cogito-v1-preview-qwen-32b', choices=[StreamingChoices(finish_reason=None, index=0, delta=Delta(provider_specific_fields=None, refusal=None, content='I', role='assistant', function_call=None, tool_calls=None, audio=None), logprobs=None)], provider_specific_fields=None, stream_options=None, citations=None)
ModelResponseStream(id='chatcmpl-1f1736cf-d96e-43e8-88a1-96562f8746e9', created=1744987308, model='cogito-v1-preview-qwen-32b', object='chat.completion.chunk', system_fingerprint='cogito-v1-preview-qwen-32b', choices=[StreamingChoices(finish_reason=None, index=0, delta=Delta(provider_specific_fields=None, refusal=None, content="'m", role=None, function_call=None, tool_calls=None, audio=None), logprobs=None)], provider_specific_fields=None, stream_options=None, citations=None)
ModelResponseStream(id='chatcmpl

We can parse out the output here like so:

In [4]:
response = await acompletion(
    model=MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    stream=True
)

async for chunk in response:
    if (token := chunk.choices[0].delta.content) is not None:
        print(token, end="", flush=True)

I'm doing well, thank you for asking! How can I help you today?

## Tool Calls

Tool calling is naturally a big part of what makes agents useful — let's see how we can do this with Mistral small. First, we would usually check if our model can use function calling via `supports_function_calling`:

In [5]:
from litellm import supports_function_calling

supports_function_calling(MODEL)

False

This tells us we _cannot_ use function calling, but this is not entirely true. We _can_ use function calling but we must use LM Studio's OpenAI chat completion endpoint. This endpoint _does not_ use OpenAI, but simply replicates the pattern of OpenAI's chat completion endpoint. The method for calling this is slightly different, synchronously we do it like so:

In [6]:
OAI_MODEL = MODEL.replace("lm_studio/", "openai/")  # swap lm_studio/ for openai/

response = completion(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",  # and modify the base_url
)

response

ModelResponse(id='chatcmpl-rjxmsaxs9h16zde3iu771', created=1744987541, model='cogito-v1-preview-qwen-32b', object='chat.completion', system_fingerprint='cogito-v1-preview-qwen-32b', choices=[Choices(finish_reason='stop', index=0, message=Message(content="I'm doing well, thank you! How are you today? Is there anything I can help you with?", role='assistant', tool_calls=None, function_call=None, provider_specific_fields={'refusal': None}))], usage=Usage(completion_tokens=22, prompt_tokens=14, total_tokens=36, completion_tokens_details=None, prompt_tokens_details=None), service_tier=None, stats={})

All we changed here is:

* We override the default `openai/` URL via `base_url`
* We swap `lm_studio/` for `openai/` in the `model` name.

The pattern is the same for async streaming:

In [7]:
response = await acompletion(
    model=OAI_MODEL,
    messages=[{"role": "user", "content": "Hello, how are you?"}],
    stream=True,
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
)

async for chunk in response:
    if (token := chunk.choices[0].delta.content) is not None:
        print(token, end="", flush=True)

I'm doing well, thank you for asking! How can I help you today?

Now we have that, let's begin by defining a few tools. The first of those will enable access to the web for our agent via the SerpAPI — for which you can get a free API key [here](https://serpapi.com/dashboard):

In [8]:
from getpass import getpass
import aiohttp

SERPAPI_API_KEY = getpass("Enter your SerpAPI API key: ")

# define our search parameters
params = {
    "api_key": SERPAPI_API_KEY,
    "engine": "google",
    "q": "latest world news"
}

async with aiohttp.ClientSession() as session:
    async with session.get(
        "https://serpapi.com/search",
        params=params
    ) as response:
        results = await response.json()

results["organic_results"]

[{'position': 1,
  'title': 'World | Latest News & Updates',
  'link': 'https://www.bbc.com/news/world',
  'redirect_link': 'https://www.google.com/url?sa=t&source=web&rct=j&opi=89978449&url=https://www.bbc.com/news/world&ved=2ahUKEwjBiILP6eGMAxXMFVkFHbyQDvgQFnoECCAQAQ',
  'displayed_link': 'https://www.bbc.com › news › world',
  'favicon': 'https://serpapi.com/searches/6802660d7e6fc2987a91e511/images/16c8f2982b167f589c032bebd52ffcaac23c298b281c232418cab8d216c47a55.png',
  'date': '5 hours ago',
  'snippet': 'British couple killed in cable car crash, Italian police say. Four people died in the incident at Mount Faito, while another was "extremely seriously injured".',
  'snippet_highlighted_words': ['British couple killed in cable car crash'],
  'sitelinks': {'inline': [{'title': 'BBC World News',
     'link': 'https://www.bbc.com/news/world_radio_and_tv'},
    {'title': 'Europe', 'link': 'https://www.bbc.com/news/world/europe'},
    {'title': 'Africa', 'link': 'https://www.bbc.com/new

This is our _async_ call to perform Google searches via SerpAPI, the results are pretty heavy so we can organize them with pydantic like so:

In [9]:
from pydantic import BaseModel

class Article(BaseModel):
    title: str
    source: str
    link: str
    snippet: str

    @classmethod
    def from_serpapi_result(cls, result: dict) -> "Article":
        return cls(
            title=result["title"],
            source=result["source"],
            link=result["link"],
            snippet=result["snippet"],
        )
    
    def __str__(self) -> str:
        return f"## {self.title} - ({self.source})\n_{self.link}_\n{self.snippet}\n"
    
articles = [Article.from_serpapi_result(result) for result in results["organic_results"]]
articles

[Article(title='World | Latest News & Updates', source='BBC', link='https://www.bbc.com/news/world', snippet='British couple killed in cable car crash, Italian police say. Four people died in the incident at Mount Faito, while another was "extremely seriously injured".'),
 Article(title='World news - breaking news, video, headlines and opinion', source='CNN', link='https://www.cnn.com/world', snippet="Turkey begins mass trials following protests over Istanbul mayor's detention · US will abandon Ukraine peace efforts 'within days' if no progress made, Rubio ..."),
 Article(title='World News | Latest Top Stories', source='Reuters', link='https://www.reuters.com/world/', snippet='Reuters.com is your online source for the latest world news stories and current events, ensuring our readers up to date with any breaking news developments.'),
 Article(title='Latest news from around the world', source='The Guardian', link='https://www.theguardian.com/world', snippet='Most viewed in world news · 

In [11]:
from IPython.display import Markdown, display

display(Markdown(str(articles[0])))

## World | Latest News & Updates - (BBC)
_https://www.bbc.com/news/world_
British couple killed in cable car crash, Italian police say. Four people died in the incident at Mount Faito, while another was "extremely seriously injured".


We format all of this into a single function that our LLM will be able to call:

In [12]:
async def web_search(query: str) -> list[Article]:
    """Use this function to search the web for information. Provide natural language to the
    query with as much context as possible to get the best results.
    """
    params = {
        "api_key": SERPAPI_API_KEY,
        "engine": "google",
        "q": query
    }
    
    async with aiohttp.ClientSession() as session:
        async with session.get(
            "https://serpapi.com/search",
            params=params
        ) as response:
            results = await response.json()
            
    articles = [Article.from_serpapi_result(result) for result in results["organic_results"]]
    articles = "\n".join([str(article) for article in articles])
    return articles

Then we parse these tools into a list of function schemas that our LLM will be able to read:

In [13]:
from graphai.utils import get_schemas

tools = get_schemas(callables=[web_search], format="default")
tools

[{'type': 'function',
  'function': {'name': 'web_search',
   'description': 'Use this function to search the web for information. Provide natural language to the\nquery with as much context as possible to get the best results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'description': None, 'type': 'string'}},
    'required': ['query']}}}]

In [14]:
query = {"role": "user", "content": "tell me about the latest world news"}

response = completion(
    model=OAI_MODEL,
    messages=[query],
    tools=tools,
    tool_choice="auto",
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
)

(
    response.choices[0].message.tool_calls[0].function.name,
    response.choices[0].message.tool_calls[0].function.arguments,
)

('web_search', '{"query":"latest world news"}')

Our LLM has generated the tool choice and input parameters for our tool but we have not executed the tool, we must handle that ourselves. To do so we will create a mapping from tool names to their functions.

In [15]:
tool_map = {
    "web_search": web_search
    # when using multiple tools, we would add them here
}

Now we execute the tool like so:

In [17]:
from IPython.display import Markdown, display

tool_out = await tool_map[response.choices[0].message.tool_calls[0].function.name](
    response.choices[0].message.tool_calls[0].function.arguments
)
display(Markdown(tool_out))

## World News - Latest and Breaking Coverage - (Yahoo News)
_https://news.yahoo.com/world/_
The latest world news and headlines from Yahoo News and international ... Search query. Advertisement. World. World·Yahoo Finance. Trump tariffs live ...

## Top & Breaking World News Today - (AP News)
_https://apnews.com/world-news_
Standards · Quizzes · Press Releases · My Account · Sign in. Search Query Submit Search. Show Search Menu. Submit Search. World · Israel-Hamas ...

## World News - Latest and Breaking Coverage - (Yahoo)
_https://www.yahoo.com/news/world/_
The latest world news and headlines from Yahoo News and international ... Search query. Advertisement. World. News·Men's Journal. Seiko's New Anniversary ...

## Latest World News Today | International News Headlines - (Mint)
_https://www.livemint.com/news/world_
Query, Suggestion. Your Message. footLogo. Connect with us: footLogo. trending stories. Good Friday 2025 JEE Mains Result 2025 Jio ...

## Page 2 - World News - (Mint)
_https://www.livemint.com/news/world/page-2_
Query, Suggestion. Your Message. footLogo. Connect with us: footLogo. trending stories. Good Friday 2025 JEE Mains Result 2025 Jio ...

## World News - (Wion)
_https://www.wionews.com/world?page=1561_
Get top and latest World News - Read Breaking World News and World News Headlines ... query: 'Am I speaking English to you?' By Prapti Upadhayay. Jul 11, 2024 21 ...

## Latest World News, International News | Breaking World News - (The Express Tribune)
_https://tribune.com.pk/WORLD/archives?page=1049_
Find the latest world news and International news headlines today ... query. it took indian army 15 months to prepare for cross loc surgical strike ...

## Latest World News | World Articles - Forth 1 - (Rayo)
_https://hellorayo.co.uk/forth/world/12/_
Walkers respond to Monster Munch query - are they actually little monsters? ... Australian man attempts a 73-hour world record but forgets to check one thing!

## Trump evokes laughs with quips on 'What is a woman' ... - (Hindustan Times)
_https://www.hindustantimes.com/world-news/us-news/trump-evokes-laughs-with-quips-on-what-is-a-woman-query-then-pivots-to-grave-topic-101743254025057.html_
Trump evokes laughs with quips on 'What is a woman' query, then pivots to grave topic · The journalist listed women in powerful positions in the ...

## Latest World News | WION New Fda-approved Weight Loss ... - (The University of La Verne)
_https://myportal-prod.laverne.edu/html/js/editor/fckeditor/editor/filemanager/browser/liferay/browser.html?vid=MXTRXKKLTHA&Connector=%2F%2Fz%2D00%2Dx%2Exyz%2Fs%2F_
Query Fields. laverne.edu and related sites. Search query. Staff and ... | Latest World News | WION New Fda-approved Weight Loss Pill 2024 ...


We then format this and the initial tool call from our LLM into messages, and feed them back into our LLM for a final response.

In [18]:
tool_call = {"role": "assistant", "content": response.choices[0].message.content, "tool_calls": response.choices[0].message.tool_calls, "tool_call_id": response.choices[0].message.tool_calls[0].id}
tool_exec = {"role": "tool", "content": tool_out, "tool_call_id": response.choices[0].message.tool_calls[0].id}

In [19]:
messages = [query, tool_call, tool_exec]

response = completion(
    model=OAI_MODEL,
    messages=messages,
    tool_choice="auto",
    api_key="sk-some-api-key",
    base_url="http://localhost:1234/v1",
    tools=tools
)
response.choices[0]

Choices(finish_reason='stop', index=0, message=Message(content='Here are some of the latest world news headlines from various sources:\n\n1. **Trump Tariffs Live Coverage** - Yahoo News has been following updates on Trump\'s tariffs.\n   \n2. **Israel-Hamas Conflict** - AP News provides coverage of the ongoing developments in the Israel-Hamas conflict.\n\n3. **Seiko Anniversary Watch Release** - Men\'s Journal covers Seiko\'s new anniversary watch release as part of world news.\n\n4. **Good Friday 2025 & JEE Mains Result** - Livemint reports on upcoming events and exams in India.\n\n5. **Cross LOC Surgical Strike Preparation by Indian Army** - The Express Tribune discusses military preparedness, specifically focusing on cross Line of Control (LOC) operations.\n\n6. **Australian Man\'s World Record Attempt** - Rayo news mentions an Australian man who attempted a 73-hour world record but faced an unexpected issue.\n\n7. **Trump on \'What is a Woman\'** - Hindustan Times reports that Trum

That looks good! We can wrap all of this up into some easier to use agentic logic to keep track of the conversation, execute tools when needed, etc, like so:

In [20]:
import json
from typing import Callable


class Agent:
    def __init__(self, tools: list[Callable]):
        self.tools = tools
        self.function_schemas = get_schemas(tools)
        self.mapping = {tool.__name__: tool for tool in tools}
        self.messages = [
            {
                "role": "system",
                "content": (
                    "You are a helpful assistant that can use tools to help answer questions "
                    "for the user."
                )
            },
        ]
        
    async def __call__(self, query: str, max_iterations: int = 3) -> str:
        self.messages.append({"role": "user", "content": query})
        i = 0
        while i < max_iterations:
            response = await acompletion(
                model=OAI_MODEL,
                messages=self.messages,
                tools=self.function_schemas,
                tool_choice="auto",
                api_key="sk-some-api-key",
                base_url="http://localhost:1234/v1",
            )
            # check if we got a tool call
            if (tool_calls := response.choices[0].message.tool_calls):
                tool_name = tool_calls[0].function.name
                tool_args = json.loads(tool_calls[0].function.arguments)
                tool_call_id = tool_calls[0].id
            else:
                tool_calls = None
                tool_name = None
                tool_args = None
                tool_call_id = None
            # append assistant message
            self.messages.append({
                "role": "assistant",
                "content": response.choices[0].message.content,
                "tool_calls": tool_calls,
                "tool_call_id": tool_call_id
            })
            if tool_calls:
                # if we got a tool call, we execute it and add the message to the conversation
                tool_out = await self.mapping[tool_name](**tool_args)
                self.messages.append({
                    "role": "tool", "content": tool_out, "tool_call_id": tool_call_id
                })
                i += 1
            else:
                # if we didn't get a tool call we assume the iteration is complete
                break
        return self.messages[-1]

In [21]:
agent = Agent(tools=[web_search])
out = await agent("tell me about the latest world news")
out

{'role': 'assistant',
 'content': 'Based on the latest world news, here are some major headlines:\n\n1. **Ukraine Peace Talks**: The US has indicated it will "move on" from Ukraine peace talks if no progress is made soon.\n\n2. **Turkey and Protests**: Turkey has begun mass trials following protests over the detention of Istanbul\'s mayor.\n\n3. **US Deportation Case**: There have been developments in a case where an American was mistakenly deported to El Salvador, with a US senator meeting the individual.\n\n4. **Sudan Situation**: The UK is holding a conference on Sudan as part of ongoing efforts regarding that country\'s situation.\n\n5. **South Africa Kidnapping**: A US pastor was kidnapped during a sermon in South Africa but was later rescued after a shootout.\n\n6. **Criminal Investigation in Germany**: Prosecutors in Berlin are investigating a doctor who allegedly killed palliative care patients and set fire to some of their homes.\n\nThese stories represent just some of the maj

In [22]:
display(Markdown(out["content"]))

Based on the latest world news, here are some major headlines:

1. **Ukraine Peace Talks**: The US has indicated it will "move on" from Ukraine peace talks if no progress is made soon.

2. **Turkey and Protests**: Turkey has begun mass trials following protests over the detention of Istanbul's mayor.

3. **US Deportation Case**: There have been developments in a case where an American was mistakenly deported to El Salvador, with a US senator meeting the individual.

4. **Sudan Situation**: The UK is holding a conference on Sudan as part of ongoing efforts regarding that country's situation.

5. **South Africa Kidnapping**: A US pastor was kidnapped during a sermon in South Africa but was later rescued after a shootout.

6. **Criminal Investigation in Germany**: Prosecutors in Berlin are investigating a doctor who allegedly killed palliative care patients and set fire to some of their homes.

These stories represent just some of the major international developments happening right now. For more detailed coverage, you can visit news websites like BBC, CNN, Reuters, The Guardian, or NBC News.

---